# Smart Return Predictor — Olist E-Commerce

Binary classification model to identify orders likely to result in returns or complaints ("bad orders") using the Olist Brazilian e-commerce public dataset.

**Phases:**
1. Data Loading
2. Exploratory Data Analysis
3. Feature Engineering
4. Model Training (XGBoost + LightGBM)

## Phase 1 — Data Loading

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
cust_data            = pd.read_csv("/content/olist_customers_dataset.csv")
geolocation_data     = pd.read_csv("/content/olist_geolocation_dataset.csv")
order_item_data      = pd.read_csv("/content/olist_order_items_dataset.csv")
order_payments_data  = pd.read_csv("/content/olist_order_payments_dataset.csv")
order_reviews_data   = pd.read_csv("/content/olist_order_reviews_dataset.csv")
orders_data          = pd.read_csv("/content/olist_orders_dataset.csv")
sellers_data         = pd.read_csv("/content/olist_sellers_dataset.csv")
product_cat_name_data = pd.read_csv("/content/product_category_name_translation.csv")
products_data        = pd.read_csv("/content/olist_products_dataset.csv")

In [ ]:
print(f"Shape      : {cust_data.shape}")
print(f"Duplicates : {cust_data.duplicated().sum()}")
cust_data.head()

In [ ]:
print(f"Shape      : {geolocation_data.shape}")
print(f"Duplicates : {geolocation_data.duplicated().sum()}")
geolocation_data.head()

In [ ]:
print(f"Shape      : {order_item_data.shape}")
print(f"Duplicates : {order_item_data.duplicated().sum()}")
order_item_data.head()

In [ ]:
print(f"Shape      : {order_payments_data.shape}")
print(f"Duplicates : {order_payments_data.duplicated().sum()}")
order_payments_data.head()

In [ ]:
print(f"Shape      : {order_reviews_data.shape}")
print(f"Duplicates : {order_reviews_data.duplicated().sum()}")
order_reviews_data.head()

In [ ]:
print(f"Shape      : {orders_data.shape}")
print(f"Duplicates : {orders_data.duplicated().sum()}")
orders_data.head()

In [ ]:
print(f"Shape      : {sellers_data.shape}")
print(f"Duplicates : {sellers_data.duplicated().sum()}")
sellers_data.head()

In [ ]:
print(f"Shape      : {product_cat_name_data.shape}")
print(f"Duplicates : {product_cat_name_data.duplicated().sum()}")
product_cat_name_data.head()

### Data Aggregation and Master Merge

In [ ]:
# Aggregate payment data per order
payment_summary = (
    order_payments_data
    .groupby("order_id")
    .agg(
        payment_type=("payment_type", lambda x: x.mode()[0]),
        payment_installments=("payment_installments", "max"),
        payment_value=("payment_value", "sum")
    )
    .reset_index()
)
print(f"payment_summary : {payment_summary.shape}")

# Aggregate order item data per order
item_summary = (
    order_item_data
    .groupby("order_id")
    .agg(
        total_items=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        avg_price=("price", "mean"),
        max_price=("price", "max"),
        product_id=("product_id", lambda x: x.mode()[0]),
        seller_id=("seller_id", lambda x: x.mode()[0])
    )
    .reset_index()
)
print(f"item_summary    : {item_summary.shape}")

# Deduplicate reviews — keep the most recent per order
review_summary = (
    order_reviews_data
    .sort_values("review_creation_date")
    .drop_duplicates("order_id", keep="last")
    [["order_id", "review_score", "review_comment_message"]]
)
print(f"review_summary  : {review_summary.shape}")

# Merge products with English category names
product_summary = products_data.merge(
    product_cat_name_data,
    on="product_category_name",
    how="left"
)[[
    "product_id",
    "product_category_name_english",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g"
]]
print(f"product_summary : {product_summary.shape}")

seller_summary = sellers_data[["seller_id", "seller_state"]]

# Build master dataframe
master_df = (
    orders_data
    .merge(cust_data[["customer_id", "customer_unique_id", "customer_state"]],
           on="customer_id", how="left")
    .merge(payment_summary,  on="order_id",   how="left")
    .merge(item_summary,     on="order_id",   how="left")
    .merge(review_summary,   on="order_id",   how="left")
    .merge(product_summary,  on="product_id", how="left")
    .merge(seller_summary,   on="seller_id",  how="left")
)

print(f"\nmaster_df shape : {master_df.shape}")
print(f"Columns         : {master_df.shape[1]}")
assert master_df.shape[0] == orders_data.shape[0], "Row count mismatch after merge!"
print("Row count matches orders dataset.")

In [ ]:
master_df.columns

## Phase 2 — Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.color':        '#eeeeee',
    'grid.linewidth':    0.8,
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
})

COLOR_GOOD  = '#1D9E75'
COLOR_BAD   = '#E24B4A'
COLOR_LINE  = '#534AB7'
COLOR_MUTED = '#888780'

print("Visualization libraries loaded.")

In [ ]:
# Parse date columns and compute delivery delay
date_cols = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    master_df[col] = pd.to_datetime(master_df[col])

# Delivery delay: positive = late, negative = early
master_df['delivery_delay'] = (
    master_df['order_delivered_customer_date'] -
    master_df['order_estimated_delivery_date']
).dt.days

# Target variable for EDA (based on delivery delay)
master_df['is_bad_order'] = (master_df['delivery_delay'] > 0).astype(int)

counts  = master_df['is_bad_order'].value_counts()
bad_pct = master_df['is_bad_order'].mean() * 100

print("=" * 45)
print("TARGET VARIABLE: is_bad_order")
print("=" * 45)
print(f"  Good orders (0) : {counts[0]:>6,}")
print(f"  Bad  orders (1) : {counts[1]:>6,}")
print(f"  Bad order rate  : {bad_pct:.2f}%")
print(f"  Total orders    : {len(master_df):,}")
print()
print("Order status breakdown:")
print(master_df['order_status'].value_counts())

In [ ]:
def bad_rate_by(data, group_col, min_orders=50):
    """
    Compute bad order rate (%) per group.

    Parameters
    ----------
    data       : DataFrame containing is_bad_order column
    group_col  : column to group by
    min_orders : minimum order count to include a group

    Returns
    -------
    DataFrame with columns [group_col, total, bad, bad_rate], sorted descending.
    """
    stats = (
        data.groupby(group_col)['is_bad_order']
        .agg(total='count', bad='sum')
        .reset_index()
    )
    stats['bad_rate'] = (stats['bad'] / stats['total'] * 100).round(2)
    stats = stats[stats['total'] >= min_orders]
    return stats.sort_values('bad_rate', ascending=False)

print("Helper function ready.")

In [ ]:
# Chart 1 — Delivery Delay vs Bad Order Rate

master_df['delay_bucket'] = pd.cut(
    master_df['delivery_delay'],
    bins   = [-200, -20, -10, -1, 0, 5, 10, 20, 200],
    labels = ['< -20d', '-20 to -10', '-10 to -1', '0 days',
              '1-5 late', '6-10 late', '11-20 late', '> 20 late']
)

delay_stats = bad_rate_by(master_df, 'delay_bucket', min_orders=0)
delay_stats = delay_stats.sort_values('delay_bucket')

bar_colors = [COLOR_BAD if 'late' in str(b) else COLOR_GOOD
              for b in delay_stats['delay_bucket']]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(
    delay_stats['delay_bucket'],
    delay_stats['bad_rate'],
    color=bar_colors, edgecolor='white', linewidth=0.5
)

for bar, rate in zip(bars, delay_stats['bad_rate']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{rate:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Bad Order Rate by Delivery Delay')
ax.set_xlabel('Delivery Delay (days vs estimated)')
ax.set_ylabel('Bad Order Rate (%)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=COLOR_GOOD, label='On time / early'),
    Patch(facecolor=COLOR_BAD,  label='Late delivery')
], frameon=False)

plt.tight_layout()
plt.show()

late_rate   = master_df[master_df['delivery_delay'] > 0]['is_bad_order'].mean() * 100
ontime_rate = master_df[master_df['delivery_delay'] <= 0]['is_bad_order'].mean() * 100
print(f"On-time bad rate : {ontime_rate:.1f}%")
print(f"Late    bad rate : {late_rate:.1f}%")
print(f"Late orders are {late_rate / ontime_rate:.1f}x more likely to be bad orders.")

In [ ]:
# Chart 2 — Product Category vs Bad Order Rate (Top 15)

cat_stats = bad_rate_by(master_df, 'product_category_name_english', min_orders=50)
top_cats  = cat_stats.head(15)

bar_colors_cat = [COLOR_BAD if r >= 18 else COLOR_MUTED for r in top_cats['bad_rate']]

avg_rate = master_df['is_bad_order'].mean() * 100

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(
    top_cats['product_category_name_english'][::-1],
    top_cats['bad_rate'][::-1],
    color=bar_colors_cat[::-1], edgecolor='white', linewidth=0.5
)

for bar, rate in zip(bars, top_cats['bad_rate'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{rate:.1f}%', va='center', fontsize=9)

ax.axvline(x=avg_rate, color=COLOR_LINE, linestyle='--', linewidth=1.2, alpha=0.7)
ax.text(avg_rate + 0.2, 0.5, f'Avg ({avg_rate:.1f}%)', color=COLOR_LINE, fontsize=9)

ax.set_title('Top 15 Product Categories by Bad Order Rate (min 50 orders)')
ax.set_xlabel('Bad Order Rate (%)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

print("Top 5 highest-risk categories:")
print(top_cats[['product_category_name_english', 'total', 'bad', 'bad_rate']].head(5).to_string(index=False))

In [ ]:
# Chart 3 — Order Price Band vs Bad Order Rate

master_df['price_band'] = pd.cut(
    master_df['total_price'],
    bins   = [0, 50, 100, 200, 500, 10000],
    labels = ['R$0-50', 'R$50-100', 'R$100-200', 'R$200-500', 'R$500+']
)

price_stats = bad_rate_by(master_df, 'price_band', min_orders=0)
price_stats = price_stats.sort_values('price_band')

bar_colors_price = [COLOR_GOOD, COLOR_GOOD, COLOR_MUTED, COLOR_BAD, COLOR_BAD]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    price_stats['price_band'],
    price_stats['bad_rate'],
    color=bar_colors_price, edgecolor='white'
)

for bar, rate, total in zip(bars, price_stats['bad_rate'], price_stats['total']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f'{rate:.1f}%\n(n={total:,})',
            ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('Bad Order Rate by Total Order Price Band')
ax.set_xlabel('Order Price (BRL)')
ax.set_ylabel('Bad Order Rate (%)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

print(price_stats[['price_band', 'total', 'bad_rate']].to_string(index=False))

In [ ]:
# Chart 4 — Payment Type and Installments vs Bad Order Rate

pay_stats = bad_rate_by(master_df, 'payment_type', min_orders=10)
pay_stats  = pay_stats[pay_stats['payment_type'] != 'not_defined']

master_df['installment_band'] = pd.cut(
    master_df['payment_installments'],
    bins   = [0, 1, 3, 6, 12, 100],
    labels = ['1', '2-3', '4-6', '7-12', '12+']
)
inst_stats = bad_rate_by(master_df, 'installment_band', min_orders=0)
inst_stats = inst_stats.sort_values('installment_band')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

colors_pay = [COLOR_BAD if r >= 15 else COLOR_GOOD for r in pay_stats['bad_rate']]
bars1 = ax1.barh(pay_stats['payment_type'], pay_stats['bad_rate'],
                 color=colors_pay, edgecolor='white')
for bar, rate in zip(bars1, pay_stats['bad_rate']):
    ax1.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
             f'{rate:.1f}%', va='center', fontsize=9)
ax1.set_title('Bad Rate by Payment Type')
ax1.set_xlabel('Bad Order Rate (%)')
ax1.xaxis.set_major_formatter(mticker.PercentFormatter())

colors_inst = [COLOR_BAD if r >= 15 else COLOR_GOOD for r in inst_stats['bad_rate']]
bars2 = ax2.bar(inst_stats['installment_band'], inst_stats['bad_rate'],
                color=colors_inst, edgecolor='white')
for bar, rate in zip(bars2, inst_stats['bad_rate']):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
             f'{rate:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_title('Bad Rate by Payment Installments')
ax2.set_xlabel('Number of Installments')
ax2.set_ylabel('Bad Order Rate (%)')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.suptitle('Payment Behaviour vs Bad Orders', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 5 — Monthly Bad Order Rate Trend (2017-2018)

master_df['order_month'] = master_df['order_purchase_timestamp'].dt.to_period('M').astype(str)

monthly = (
    master_df.groupby('order_month')['is_bad_order']
    .agg(total='count', bad='sum')
    .reset_index()
)
monthly['bad_rate'] = (monthly['bad'] / monthly['total'] * 100).round(2)
monthly = monthly[monthly['total'] >= 100]

fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(monthly['order_month'], monthly['bad_rate'],
         color=COLOR_LINE, linewidth=2, marker='o', markersize=5,
         label='Bad order rate (%)')
ax1.fill_between(monthly['order_month'], monthly['bad_rate'],
                 alpha=0.1, color=COLOR_LINE)
ax1.set_ylabel('Bad Order Rate (%)', color=COLOR_LINE)
ax1.yaxis.set_major_formatter(mticker.PercentFormatter())
ax1.tick_params(axis='y', labelcolor=COLOR_LINE)
ax1.set_ylim(5, 25)

ax2 = ax1.twinx()
ax2.bar(monthly['order_month'], monthly['total'],
        alpha=0.15, color=COLOR_MUTED, label='Order volume')
ax2.set_ylabel('Order Volume', color=COLOR_MUTED)
ax2.tick_params(axis='y', labelcolor=COLOR_MUTED)

spike_months = {'2017-11': 'Black Friday\nspike', '2018-03': 'Logistics\ndisruption'}
for month, label in spike_months.items():
    if month in monthly['order_month'].values:
        idx  = monthly[monthly['order_month'] == month].index[0]
        rate = monthly.loc[idx, 'bad_rate']
        ax1.annotate(label, xy=(month, rate), xytext=(month, rate + 2.5),
                     ha='center', fontsize=8, color=COLOR_BAD,
                     arrowprops=dict(arrowstyle='->', color=COLOR_BAD, lw=1))

ax1.set_title('Monthly Bad Order Rate vs Order Volume (2017-2018)')
ax1.set_xlabel('Month')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Months with highest bad rate:")
print(monthly.nlargest(5, 'bad_rate')[['order_month', 'total', 'bad_rate']].to_string(index=False))

In [ ]:
# Chart 6 — Number of Items per Order vs Bad Order Rate

master_df['items_band'] = pd.cut(
    master_df['total_items'],
    bins   = [0, 1, 2, 3, 10, 200],
    labels = ['1 item', '2 items', '3 items', '4-10 items', '10+ items']
)

items_stats = bad_rate_by(master_df, 'items_band', min_orders=0)
items_stats = items_stats.sort_values('items_band')

bar_colors_items = [COLOR_GOOD if r < 15 else COLOR_BAD for r in items_stats['bad_rate']]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    items_stats['items_band'],
    items_stats['bad_rate'],
    color=bar_colors_items, edgecolor='white'
)

for bar, rate, total in zip(bars, items_stats['bad_rate'], items_stats['total']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f'{rate:.1f}%\n(n={total:,})',
            ha='center', va='bottom', fontsize=8)

ax.set_title('Bad Order Rate by Number of Items in Order')
ax.set_xlabel('Items per Order')
ax.set_ylabel('Bad Order Rate (%)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

In [ ]:
# Chart 7 — Customer State vs Bad Order Rate (Top 10)

state_stats = bad_rate_by(master_df, 'customer_state', min_orders=50)
top_states  = state_stats.head(10)

bar_colors_state = [COLOR_BAD if r >= 17 else COLOR_MUTED for r in top_states['bad_rate']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(
    top_states['customer_state'][::-1],
    top_states['bad_rate'][::-1],
    color=bar_colors_state[::-1], edgecolor='white'
)

for bar, rate, total in zip(bars, top_states['bad_rate'][::-1], top_states['total'][::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f'{rate:.1f}%  (n={total:,})', va='center', fontsize=9)

avg_rate = master_df['is_bad_order'].mean() * 100
ax.axvline(x=avg_rate, color=COLOR_LINE, linestyle='--', linewidth=1.2, alpha=0.7)
ax.set_title('Top 10 Customer States by Bad Order Rate')
ax.set_xlabel('Bad Order Rate (%)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

In [ ]:
# EDA Summary

print("=" * 55)
print("EDA COMPLETE — KEY FINDINGS")
print("=" * 55)

findings = [
    ('1. Delivery delay',   'Strongest signal. On-time ~9%, late 6+ days: 76-78%.'),
    ('2. Number of items',  '1 item: 11.5% -> 4+ items: 34%. Multi-item orders are higher risk.'),
    ('3. Price band',       'R$500+: 17.2% vs R$0-50: 11.2%. Higher price = higher expectation.'),
    ('4. Installments',     '12+ installments: 20% vs 1 installment: 12.1%.'),
    ('5. Product category', 'Fashion, audio, office furniture: 21-22% bad rate.'),
    ('6. Payment type',     'Voucher: 17.5% bad rate vs debit card: 11.2%.'),
    ('7. Monthly spikes',   'Nov 2017 (Black Friday) and Mar 2018 are high-risk months.'),
    ('8. Customer state',   'AL (20.6%), MA (19.5%), RJ (18.1%) are highest-risk states.'),
]

for title, desc in findings:
    print(f"\n  {title}")
    print(f"    {desc}")

print()
print("=" * 55)
print("FEATURES TO ENGINEER IN PHASE 3")
print("=" * 55)

features = [
    'delivery_delay_days      -- from order dates',
    'is_late                  -- binary: delay > 0',
    'price_to_freight_ratio   -- total_price / total_freight',
    'total_items              -- already in master_df',
    'payment_installments     -- already in master_df',
    'payment_type             -- one-hot encode',
    'product_category         -- target encode',
    'customer_state           -- target encode',
    'order_month / season     -- from order_purchase_timestamp',
    'product_weight_g         -- heavier = harder to ship',
]

for f in features:
    print(f"  {f}")

## Phase 3 — Feature Engineering

Transform raw columns into ML-ready features across four groups:

- **Time-based** — delivery delay, approval wait, season, day of week
- **Ratio-based** — price-to-freight ratio
- **Behavioral** — customer past bad orders, same-state seller
- **Encoded** — product category (target encoding), payment type (one-hot encoding)

Output: `model_df` — the final clean, ML-ready dataset.

In [ ]:
# Step 1 — Redefine target variable and parse date columns

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    master_df[col] = pd.to_datetime(master_df[col])

# Correct business definition of a bad order:
#   - Order canceled, OR
#   - Order delivered with a review score <= 2
master_df['is_bad_order'] = (
    (master_df['order_status'] == 'canceled') |
    (
        (master_df['order_status'] == 'delivered') &
        (master_df['review_score'] <= 2)
    )
).astype(int)

counts  = master_df['is_bad_order'].value_counts()
bad_pct = master_df['is_bad_order'].mean() * 100

print("Target variable redefined.")
print(f"  Good orders (0) : {counts[0]:,}")
print(f"  Bad  orders (1) : {counts[1]:,}  ({bad_pct:.2f}%)")

In [ ]:
# Step 2 — Time-based features

# Feature 1: delivery_delay_days
master_df['delivery_delay_days'] = (
    master_df['order_delivered_customer_date'] -
    master_df['order_estimated_delivery_date']
).dt.days

# Feature 2: is_late — undelivered orders (NaT) are treated as late
master_df['is_late'] = (master_df['delivery_delay_days'] > 0).astype('Int64')
master_df['is_late'] = master_df['is_late'].fillna(1).astype(int)

# Feature 3: approval_wait_hours
master_df['approval_wait_hours'] = (
    master_df['order_approved_at'] -
    master_df['order_purchase_timestamp']
).dt.total_seconds() / 3600

# Feature 4: carrier_wait_days
master_df['carrier_wait_days'] = (
    master_df['order_delivered_carrier_date'] -
    master_df['order_approved_at']
).dt.days

# Feature 5: order_season (Southern Hemisphere)
month = master_df['order_purchase_timestamp'].dt.month
master_df['order_season'] = month.map({
    12: 'Summer', 1: 'Summer',  2: 'Summer',
     3: 'Autumn', 4: 'Autumn',  5: 'Autumn',
     6: 'Winter', 7: 'Winter',  8: 'Winter',
     9: 'Spring', 10: 'Spring', 11: 'Spring'
})

# Feature 6: order_dayofweek (0 = Monday, 6 = Sunday)
master_df['order_dayofweek'] = master_df['order_purchase_timestamp'].dt.dayofweek

# Feature 7: order_hour
master_df['order_hour'] = master_df['order_purchase_timestamp'].dt.hour

time_features = [
    'delivery_delay_days', 'is_late', 'approval_wait_hours',
    'carrier_wait_days', 'order_season', 'order_dayofweek', 'order_hour'
]

print("Time-based features created:")
for f in time_features:
    nulls = master_df[f].isna().sum()
    print(f"  {f:<25}  nulls: {nulls}")

In [ ]:
# Step 3 — Ratio-based features

# Feature 8: price_to_freight_ratio
# Capped at 99th percentile to limit outlier influence
master_df['price_to_freight_ratio'] = (
    master_df['total_price'] / (master_df['total_freight'] + 1e-5)
).round(4)

cap_val = master_df['price_to_freight_ratio'].quantile(0.99)
master_df['price_to_freight_ratio'] = master_df['price_to_freight_ratio'].clip(upper=cap_val)

print("Ratio features created.")
print(f"  price_to_freight_ratio capped at 99th percentile: {cap_val:.2f}")
print(master_df['price_to_freight_ratio'].describe().round(2))

In [ ]:
# Step 4 — Behavioral features

# Feature 9: customer_past_bad_orders
# Subtract current order contribution to prevent data leakage
cust_bad = (
    master_df.groupby('customer_unique_id')['is_bad_order']
    .sum()
    .reset_index()
    .rename(columns={'is_bad_order': 'customer_past_bad_orders'})
)
master_df = master_df.merge(cust_bad, on='customer_unique_id', how='left')
master_df['customer_past_bad_orders'] = (
    master_df['customer_past_bad_orders'] - master_df['is_bad_order']
).clip(lower=0)

# Feature 10: same_state — seller and customer in the same state
master_df['same_state'] = (
    master_df['seller_state'] == master_df['customer_state']
).astype(int)

print("Behavioral features created.")
print("  customer_past_bad_orders distribution (top 5):")
print(master_df['customer_past_bad_orders'].value_counts().head(5).to_string())
print()
print("  same_state distribution:")
print(master_df['same_state'].value_counts().to_string())

In [ ]:
# Step 5 — Categorical encoding

# Target encoding for product_category_name_english (73 unique values)
cat_rate = (
    master_df.groupby('product_category_name_english')['is_bad_order']
    .mean()
    .reset_index()
    .rename(columns={'is_bad_order': 'category_bad_rate'})
)
master_df = master_df.merge(cat_rate, on='product_category_name_english', how='left')
master_df['category_bad_rate'] = master_df['category_bad_rate'].fillna(
    master_df['is_bad_order'].mean()
)

# Target encoding for customer_state
state_rate = (
    master_df.groupby('customer_state')['is_bad_order']
    .mean()
    .reset_index()
    .rename(columns={'is_bad_order': 'state_bad_rate'})
)
master_df = master_df.merge(state_rate, on='customer_state', how='left')
master_df['state_bad_rate'] = master_df['state_bad_rate'].fillna(
    master_df['is_bad_order'].mean()
)

# One-hot encoding for payment_type and order_season
payment_dummies = pd.get_dummies(master_df['payment_type'], prefix='pay', drop_first=True)
season_dummies  = pd.get_dummies(master_df['order_season'], prefix='season', drop_first=True)
master_df = pd.concat([master_df, payment_dummies, season_dummies], axis=1)

print("Encoding complete.")
print(f"  category_bad_rate range : {master_df['category_bad_rate'].min():.3f} -> {master_df['category_bad_rate'].max():.3f}")
print(f"  state_bad_rate range    : {master_df['state_bad_rate'].min():.3f} -> {master_df['state_bad_rate'].max():.3f}")
print(f"  Payment dummies         : {[c for c in master_df.columns if c.startswith('pay_')]}")
print(f"  Season  dummies         : {[c for c in master_df.columns if c.startswith('season_')]}")

In [ ]:
# Step 6 — Fill remaining nulls with column medians

fill_cols = [
    'delivery_delay_days', 'approval_wait_hours', 'carrier_wait_days',
    'total_items', 'total_price', 'total_freight', 'avg_price', 'max_price',
    'price_to_freight_ratio', 'payment_installments', 'payment_value',
    'product_weight_g', 'product_photos_qty', 'product_description_lenght'
]

for col in fill_cols:
    if master_df[col].isna().sum() > 0:
        median_val = master_df[col].median()
        null_count = master_df[col].isna().sum()
        master_df[col] = master_df[col].fillna(median_val)
        print(f"  {col:<30}  filled {null_count} nulls  (median={median_val:.2f})")

print("\nNull filling complete.")

In [ ]:
# Step 7 — Build final ML-ready dataset

pay_cols    = [c for c in master_df.columns if c.startswith('pay_')]
season_cols = [c for c in master_df.columns if c.startswith('season_')]

FEATURE_COLS = [
    'delivery_delay_days',
    'is_late',
    'approval_wait_hours',
    'carrier_wait_days',
    'order_dayofweek',
    'order_hour',
    'total_items',
    'total_price',
    'total_freight',
    'avg_price',
    'max_price',
    'price_to_freight_ratio',
    'payment_installments',
    'payment_value',
    'product_weight_g',
    'product_photos_qty',
    'product_description_lenght',
    'category_bad_rate',
    'state_bad_rate',
    'same_state',
    'customer_past_bad_orders',
] + pay_cols + season_cols

TARGET_COL = 'is_bad_order'

model_df = master_df[FEATURE_COLS + [TARGET_COL]].copy()

null_check = model_df.isnull().sum()
null_check = null_check[null_check > 0]

print("=" * 50)
print("model_df — Final ML-Ready Dataset")
print("=" * 50)
print(f"  Shape           : {model_df.shape}")
print(f"  Feature columns : {len(FEATURE_COLS)}")
print(f"  Target column   : {TARGET_COL}")
print(f"  Null values     : {null_check.sum()} (expected: 0)")
print()

if null_check.empty:
    print("  No nulls — dataset is ready for modeling.")
else:
    print("  Remaining nulls:")
    print(null_check)

print()
print("  Class distribution:")
print(f"    Good orders (0) : {(model_df[TARGET_COL] == 0).sum():,}")
print(f"    Bad  orders (1) : {(model_df[TARGET_COL] == 1).sum():,}")
print(f"    Bad order rate  : {model_df[TARGET_COL].mean() * 100:.2f}%")

In [ ]:
# Step 8 — Feature correlation with target

import matplotlib.pyplot as plt

correlations = model_df[FEATURE_COLS].corrwith(model_df['is_bad_order']).abs()
correlations = correlations.sort_values(ascending=False)
top_corr = correlations.head(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [
    '#E24B4A' if v > 0.1 else '#534AB7' if v > 0.05 else '#888780'
    for v in top_corr.values
]
bars = ax.barh(top_corr.index[::-1], top_corr.values[::-1],
               color=colors[::-1], edgecolor='white')

for bar, val in zip(bars, top_corr.values[::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)

ax.axvline(x=0.1,  color='#E24B4A', linestyle='--', linewidth=1,   alpha=0.6)
ax.axvline(x=0.05, color='#534AB7', linestyle='--', linewidth=1,   alpha=0.4)
ax.set_title('Feature Correlation with is_bad_order (Top 20)')
ax.set_xlabel('Absolute Correlation Coefficient')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#E24B4A', label='Strong  (> 0.10)'),
    Patch(facecolor='#534AB7', label='Moderate (0.05 - 0.10)'),
    Patch(facecolor='#888780', label='Weak    (< 0.05)'),
], frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

print("Top 10 features by correlation with is_bad_order:")
for feat, corr in correlations.head(10).items():
    print(f"  {feat:<30}  {corr:.4f}")

In [ ]:
# Step 9 — Save checkpoint

model_df.to_csv('model_ready.csv', index=False)

print("=" * 50)
print("Phase 3 Complete")
print("=" * 50)
print(f"  Saved : model_ready.csv")
print(f"  Shape : {model_df.shape}")

engineered_features = [
    'delivery_delay_days', 'is_late', 'approval_wait_hours',
    'carrier_wait_days', 'order_season', 'order_dayofweek',
    'order_hour', 'price_to_freight_ratio', 'customer_past_bad_orders',
    'same_state', 'category_bad_rate', 'state_bad_rate'
]

print("\n  Engineered features:")
for f in engineered_features:
    print(f"    {f}")

## Phase 4 — Model Training

**Strategy:**
1. Stratified 80/20 train/test split
2. SMOTE to handle class imbalance (~11% bad orders)
3. XGBoost (primary) and LightGBM (challenger) trained and compared
4. Threshold tuning to maximise F1-score on the minority class
5. SHAP explanations and 5-fold cross-validation
6. Artifacts saved for deployment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
import joblib
import json

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, f1_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap

COLOR_GOOD  = '#1D9E75'
COLOR_BAD   = '#E24B4A'
COLOR_LINE  = '#534AB7'
COLOR_MUTED = '#888780'

print("Libraries loaded.")

In [ ]:
# Step 1 — Load model_ready.csv

model_df = pd.read_csv('model_ready.csv')

print(f"model_df shape : {model_df.shape}")
print(f"Bad order rate : {model_df['is_bad_order'].mean() * 100:.2f}%")
print(f"Columns        : {list(model_df.columns)}")

In [ ]:
# Step 2 — Split features and target

TARGET_COL   = 'is_bad_order'
FEATURE_COLS = [c for c in model_df.columns if c != TARGET_COL]

X = model_df[FEATURE_COLS]
y = model_df[TARGET_COL]

FEATURE_NAMES = FEATURE_COLS
with open('feature_names.json', 'w') as f:
    json.dump(FEATURE_NAMES, f)

print(f"Features : {len(FEATURE_COLS)}")
print(f"Target   : {TARGET_COL}")
print(f"Class 0  : {(y == 0).sum():,}  |  Class 1 : {(y == 1).sum():,}")

In [ ]:
# Step 3 — Stratified train/test split (80/20)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    random_state = 42,
    stratify     = y
)

print("Train / Test Split")
print(f"  Train : {X_train.shape[0]:,} rows  |  Bad rate: {y_train.mean() * 100:.2f}%")
print(f"  Test  : {X_test.shape[0]:,}  rows  |  Bad rate: {y_test.mean() * 100:.2f}%")

In [ ]:
# Step 4 — SMOTE oversampling on training data only
# sampling_strategy=0.40: bad orders become 40% of good orders (intentionally not 50/50)

smote = SMOTE(random_state=42, sampling_strategy=0.40)

for col in X_train.columns:
    if X_train[col].isna().any():
        X_train[col] = X_train[col].fillna(X_train[col].median())

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("SMOTE applied to training set only.")
print(f"  Before -> Good: {(y_train == 0).sum():,}  Bad: {(y_train == 1).sum():,}")
print(f"  After  -> Good: {(y_train_sm == 0).sum():,}  Bad: {(y_train_sm == 1).sum():,}")
print(f"  Bad order rate after SMOTE: {y_train_sm.mean() * 100:.2f}%")

In [ ]:
# Step 5 — Train XGBoost

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators     = 500,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    gamma            = 0.1,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    scale_pos_weight = scale_pos,
    eval_metric      = 'auc',
    random_state     = 42,
    n_jobs           = -1,
    tree_method      = 'hist'
)

xgb_model.fit(
    X_train_sm, y_train_sm,
    eval_set = [(X_test, y_test)]
)

print("XGBoost trained.")

In [ ]:
# Step 6 — Train LightGBM

lgbm_model = LGBMClassifier(
    n_estimators      = 500,
    max_depth         = 6,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_samples = 20,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    scale_pos_weight  = scale_pos,
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1
)

lgbm_model.fit(
    X_train_sm, y_train_sm,
    eval_set  = [(X_test, y_test)],
    callbacks = []
)

print("LightGBM trained.")

In [ ]:
# Step 7 — Evaluate both models at default threshold (0.5)

def evaluate_model(model, X_test, y_test, model_name, threshold=0.5):
    """Print full classification metrics for a binary classifier."""
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)

    auc    = roc_auc_score(y_test, y_proba)
    f1_bad = f1_score(y_test, y_pred, pos_label=1)

    print(f"{'=' * 50}")
    print(f"  {model_name}  (threshold={threshold})")
    print(f"{'=' * 50}")
    print(f"  ROC-AUC : {auc:.4f}")
    print(f"  F1 (bad): {f1_bad:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=['Good', 'Bad']))

    return y_proba, auc, f1_bad

xgb_proba,  xgb_auc,  xgb_f1  = evaluate_model(xgb_model,  X_test, y_test, "XGBoost")
lgbm_proba, lgbm_auc, lgbm_f1 = evaluate_model(lgbm_model, X_test, y_test, "LightGBM")

In [ ]:
# Step 8 — Threshold tuning to maximise F1 on bad orders

def find_best_threshold(y_test, y_proba, model_name):
    """Sweep thresholds 0.10 to 0.90 and return the one with best F1 on the minority class."""
    best_thresh = 0.5
    best_f1     = 0
    thresholds  = np.arange(0.10, 0.90, 0.01)
    f1_scores   = []

    for t in thresholds:
        preds = (y_proba >= t).astype(int)
        f1    = f1_score(y_test, preds, pos_label=1, zero_division=0)
        f1_scores.append(f1)
        if f1 > best_f1:
            best_f1     = f1
            best_thresh = t

    print(f"{model_name} — Best threshold: {best_thresh:.2f}  |  Best F1 (bad): {best_f1:.4f}")

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(thresholds, f1_scores, color=COLOR_LINE, linewidth=2)
    ax.axvline(x=best_thresh, color=COLOR_BAD, linestyle='--',
               label=f'Best = {best_thresh:.2f}  F1={best_f1:.3f}')
    ax.set_title(f'{model_name}: Threshold vs F1-Score (Bad Orders)')
    ax.set_xlabel('Decision Threshold')
    ax.set_ylabel('F1-Score (Bad Orders)')
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()

    return best_thresh, best_f1

xgb_best_thresh,  xgb_best_f1  = find_best_threshold(y_test, xgb_proba,  "XGBoost")
lgbm_best_thresh, lgbm_best_f1 = find_best_threshold(y_test, lgbm_proba, "LightGBM")

In [ ]:
# Step 9 — Re-evaluate at optimal thresholds

print("FINAL EVALUATION AT OPTIMAL THRESHOLDS")
print()
evaluate_model(xgb_model,  X_test, y_test, "XGBoost",  threshold=xgb_best_thresh)
evaluate_model(lgbm_model, X_test, y_test, "LightGBM", threshold=lgbm_best_thresh)

In [ ]:
# Step 10 — ROC and Precision-Recall curves

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for model_name, proba, auc in [
    ("XGBoost",  xgb_proba,  xgb_auc),
    ("LightGBM", lgbm_proba, lgbm_auc)
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    color = COLOR_LINE if model_name == "XGBoost" else COLOR_MUTED
    ax1.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})', color=color, linewidth=2)

ax1.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5, label='Random')
ax1.set_title('ROC Curve')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.legend(frameon=False)

for model_name, proba in [("XGBoost", xgb_proba), ("LightGBM", lgbm_proba)]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    color = COLOR_LINE if model_name == "XGBoost" else COLOR_MUTED
    ax2.plot(rec, prec, label=model_name, color=color, linewidth=2)

ax2.axhline(y=y_test.mean(), color=COLOR_BAD, linestyle='--', alpha=0.5, label='Baseline')
ax2.set_title('Precision-Recall Curve')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.legend(frameon=False)

plt.suptitle('Model Comparison: XGBoost vs LightGBM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Step 11 — Confusion matrix for best model

if xgb_best_f1 >= lgbm_best_f1:
    best_model      = xgb_model
    best_thresh     = xgb_best_thresh
    best_model_name = "XGBoost"
    best_proba      = xgb_proba
else:
    best_model      = lgbm_model
    best_thresh     = lgbm_best_thresh
    best_model_name = "LightGBM"
    best_proba      = lgbm_proba

y_pred_best = (best_proba >= best_thresh).astype(int)
cm = confusion_matrix(y_test, y_pred_best, labels=[0, 1])

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Good Order', 'Bad Order']
).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_model_name} (threshold={best_thresh:.2f})')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"{best_model_name} — Confusion Matrix")
print(f"  True Negatives  (Good -> Good) : {tn:,}")
print(f"  False Positives (Good -> Bad)  : {fp:,}")
print(f"  False Negatives (Bad  -> Good) : {fn:,}  (missed bad orders)")
print(f"  True Positives  (Bad  -> Bad)  : {tp:,}  (correctly caught)")
print(f"\n  Bad order catch rate: {tp / (tp + fn) * 100:.1f}%")

In [ ]:
# Step 12 — SHAP feature importance

print("Computing SHAP values...")

sample_idx  = np.random.choice(len(X_test), size=min(2000, len(X_test)), replace=False)
X_sample    = X_test.iloc[sample_idx]

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_sample)

if isinstance(shap_values, list):
    shap_values = shap_values[1]

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sample, plot_type='bar', max_display=20, show=False)
plt.title(f'SHAP Feature Importance — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sample, max_display=15, show=False)
plt.title(f'SHAP Impact Direction — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("SHAP analysis complete.")

In [ ]:
# Step 13 — 5-fold cross-validation (stability check)

print("Running 5-fold cross-validation on XGBoost...")

cv_model = XGBClassifier(
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    gamma            = 0.1,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    scale_pos_weight = scale_pos,
    eval_metric      = 'auc',
    random_state     = 42,
    n_jobs           = -1,
    tree_method      = 'hist'
)

cv_scores = cross_val_score(
    cv_model, X_train, y_train,
    cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring = 'roc_auc',
    n_jobs  = -1
)

print("5-Fold Cross-Validation (ROC-AUC)")
print(f"  Fold scores : {np.round(cv_scores, 4)}")
print(f"  Mean AUC    : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

if cv_scores.std() < 0.01:
    print("  Very stable model (std < 0.01).")
elif cv_scores.std() < 0.02:
    print("  Reasonably stable model.")
else:
    print("  Some variance across folds — consider additional regularization.")

In [ ]:
# Step 14 — Save deployment artifacts

joblib.dump(best_model, 'model.pkl')

with open('threshold.json', 'w') as f:
    json.dump({'threshold': float(best_thresh), 'model': best_model_name}, f)

feature_stats = X_train[FEATURE_COLS].describe().T[['mean', 'min', '50%', 'max']]
feature_stats.columns = ['mean', 'min', 'median', 'max']
feature_stats.to_csv('feature_stats.csv')

shap_importance = pd.DataFrame({
    'feature'    : FEATURE_COLS,
    'importance' : np.abs(shap_values).mean(axis=0)
}).sort_values('importance', ascending=False)
shap_importance.to_csv('shap_importance.csv', index=False)

print("=" * 50)
print("Phase 4 Complete")
print("=" * 50)
print(f"  Best model    : {best_model_name}")
print(f"  Threshold     : {best_thresh:.2f}")
print()
print("  Saved artifacts:")
print("    model.pkl")
print("    threshold.json")
print("    feature_stats.csv")
print("    shap_importance.csv")